# 데이터 품질 점검 (1) — 테이블 구조와 관계

9 개 테이블을 조인해 집계했을 때 결과가 틀어지지 않는지 확인.
조인 한 번으로 행이 몇 배가 되거나 표본이 조용히 빠지는 일이 여기서 갈리므로
분석보다 먼저 진행.

**확인 항목**

- 그레인 — 한 행이 무엇 하나를 가리키는지, 그것을 유일하게 식별하는 키는 무엇인지
- 참조 무결성 — 다른 테이블에 짝이 없는 행이 있는지
- 카디널리티 — 지역·카테고리처럼 집계 기준이 될 컬럼의 값 종류
- 중복 — 같은 대상이 여러 행으로 들어온 경우와 그 성격

**키 확정 기준**

키 후보는 Kaggle 데이터셋 페이지의 컬럼 설명과 ERD를 참고해 선정.
확정은 문서가 아니라 아래 절차의 결과로 판단.
같은 절차는 컬럼 설명이 없는 테이블에도 그대로 적용 가능.

**절차**

1. 후보 키에 대해 `COUNT(*)` 와 `COUNT(DISTINCT 후보키)` 비교
2. 두 값이 다르면 `GROUP BY 후보키 HAVING COUNT(*) > 1` 로 중복 행 확인
3. 중복을 구분하는 열을 찾아 복합 키 구성
4. 확정한 키로 다른 테이블과의 짝을 확인

## 0. 연결

조회만 하므로 읽기 전용으로 연결.

In [1]:
import duckdb

con = duckdb.connect('../olist.duckdb', read_only=True)

con

## 1. orders

후보 키: `order_id`

컬럼 구성 확인.

In [2]:
con.execute("""
SELECT *
FROM orders
LIMIT 3
""").df()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


주문 상태(`order_status`)와 구매·승인·출고·배송완료·예상 배송의 5개 시각 보유.
고객은 `customer_id` 로 연결.

In [3]:
con.execute("""
SELECT COUNT(*)                  AS row_cnt,
       COUNT(DISTINCT order_id)  AS order_id_cnt
FROM orders
""").df()

,row_cnt,order_id_cnt
0,99441,99441


두 값 모두 99,441 로 동일.

그레인은 주문 1건, 키는 `order_id`. 이후 조인의 기준 테이블로 사용.

## 2. customers

후보 키: `customer_id` / `customer_unique_id`

컬럼 구성 확인.

In [4]:
con.execute("""
SELECT *
FROM customers
LIMIT 3
""").df()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


주(`customer_state`)·도시·우편번호 앞자리를 자체 보유.
이름이 비슷한 두 개의 ID 가 있어 함께 확인.

In [5]:
con.execute("""
SELECT COUNT(*)                          AS row_cnt,
       COUNT(DISTINCT customer_id)        AS customer_id_cnt,
       COUNT(DISTINCT customer_unique_id) AS customer_unique_id_cnt
FROM customers
""").df()

,row_cnt,customer_id_cnt,customer_unique_id_cnt
0,99441,99441,96096


`customer_id` 는 99,441 로 행 수와 동일. `customer_unique_id` 는 96,096.
차이 3,345 건은 같은 사람이 재주문하며 새 `customer_id` 를 받은 레코드.

그레인은 사람이 아니라 주문 시점의 고객 레코드.
`orders` 와 1:1 대응하는지 양방향으로 확인.

In [6]:
con.execute("""
SELECT (SELECT COUNT(*) FROM orders)                        AS orders_cnt,
       (SELECT COUNT(DISTINCT customer_id) FROM orders)     AS orders_customer_id_cnt,
       (SELECT COUNT(*) FROM orders AS o
         LEFT JOIN customers AS c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL)                        AS orders_without_customer,
       (SELECT COUNT(*) FROM customers AS c
         LEFT JOIN orders AS o ON c.customer_id = o.customer_id
        WHERE o.customer_id IS NULL)                        AS customers_without_order
""").df()

,orders_cnt,orders_customer_id_cnt,orders_without_customer,customers_without_order
0,99441,99441,0,0


`orders` 의 `customer_id` 도 99,441 개로 전부 서로 다르고, 짝이 없는 행은 양쪽 모두 0 건.
`orders` 와 `customers` 는 `customer_id` 로 1:1 대응.

주문 단위 분석은 `customer_id`, 사람 단위 분석은 `customer_unique_id` 로 집계.

지역 분석에 쓸 `customer_city` 와 `customer_state` 의 값 종류 확인.

In [7]:
con.execute("""
SELECT COUNT(DISTINCT customer_city)                    AS city_cnt,
       COUNT(DISTINCT customer_state)                   AS state_cnt,
       COUNT(DISTINCT (customer_city, customer_state))  AS city_state_cnt
FROM customers
""").df()

,city_cnt,state_cnt,city_state_cnt
0,4119,27,4310


주 27 개, 도시 4,119 개. `(city, state)` 조합은 4,310 개로 도시 수보다 191 개 많다.
같은 도시명이 서로 다른 주에 존재하므로 도시를 단독으로 묶으면 다른 지역이 합쳐짐.
도시 단위 집계는 `(customer_city, customer_state)` 로 수행.

지역 단위를 주로 둘지 도시로 내릴지는 규모 분포를 보고 분석 표본을 정할 때 결정.

## 3. order_items

주문에 담긴 상품을 담는 테이블. 컬럼 구성 확인.

In [8]:
con.execute("""
SELECT *
FROM order_items
LIMIT 3
""").df()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


상품(`product_id`)·판매자(`seller_id`)·금액(`price`, `freight_value`) 보유. 수량 컬럼 없음.

`order_id` 단독으로 키가 되는지 확인.

In [9]:
con.execute("""
SELECT COUNT(*)                                    AS row_cnt,
       COUNT(DISTINCT order_id)                    AS order_id_cnt,
       COUNT(DISTINCT (order_id, order_item_id))   AS composite_key_cnt
FROM order_items
""").df()

,row_cnt,order_id_cnt,composite_key_cnt
0,112650,98666,112650


행 수 112,650 에 `order_id` 는 98,666.
`order_item_id` 를 더한 복합 키는 112,650 으로 행 수와 일치.
(`order_item_id` 단독은 프리뷰에서 서로 다른 세 주문이 모두 `1` 인 것으로 이미 배제됨.)

한 주문이 몇 행까지 갖는지 확인.

In [10]:
con.execute("""
SELECT order_id,
       COUNT(*) AS item_cnt
FROM order_items
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY item_cnt DESC
LIMIT 5
""").df()

,order_id,item_cnt
0,8272b63d03f5f79c56e9e4120aec44ef,21
1,ab14fdcfbe524636d65ee38360e22ce8,20
2,1b15974a0141d54e36626dca3fdc731a,20
3,9ef13efd6949e4573a18964dd1bbe7f5,15
4,428a2f660dc84138d969ccd69a0ab6d5,15


최대 21 행. 이 21 행이 서로 다른 상품인지 펼쳐서 확인.

In [11]:
con.execute("""
SELECT order_item_id,
       product_id,
       seller_id,
       price,
       freight_value
FROM order_items
WHERE order_id = '8272b63d03f5f79c56e9e4120aec44ef'
ORDER BY order_item_id
""").df()

,order_item_id,product_id,seller_id,price,freight_value
0,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,1.2,7.89
1,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
2,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
3,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
4,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
5,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
6,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
7,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
8,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89
9,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,1.2,7.89


판매자는 하나, 상품은 3 종류. 두 상품이 각각 10 회 반복되고 나머지 하나는 1 회.
반복되는 20 행은 `price` 1.2 · `freight_value` 7.89 로 값이 동일하고,
21 번째 행만 다른 상품이라 7.8 · 6.57 로 다름.
금액은 행마다 매겨진 것이 아니라 상품에 붙은 값.

수량 컬럼이 없는 대신 같은 상품을 여러 개 사면 행이 그 개수만큼 생성.
`order_item_id` 는 상품 구분이 아니라 주문 안의 행 번호.

그레인은 주문에서 구매한 상품 1개, 키는 `(order_id, order_item_id)`.
주문 단위 지표는 `order_id` 로 집계해서 산출 — 수량 `COUNT(*)`, 상품 금액 `SUM(price)`.

운임은 아직 단정하지 않음. 위 주문에서 `freight_value` 가 행마다 달랐으나, 그것만으로는
행마다 붙는 값인지 주문 단위 값이 섞여 들어온 것인지 갈리지 않음.
주문당 한 번 붙는 값이라면 `SUM` 은 값을 행 수만큼 부풀림.
판정에 결제액이 필요하므로 `order_payments` 를 확인한 뒤 4 끝에서 다룸.

`order_id` 98,666 개는 orders 의 99,441 건에 미달.
품목 기록이 없는 주문 775 건이 어떤 주문인지 확인.

In [12]:
con.execute("""
SELECT o.order_status,
       COUNT(*) AS order_cnt,
       COUNT(*) FILTER (WHERE p.order_id IS NOT NULL) AS has_payment,
       COUNT(*) FILTER (WHERE r.order_id IS NOT NULL) AS has_review
FROM orders AS o
LEFT JOIN (SELECT DISTINCT order_id FROM order_items)    AS i ON o.order_id = i.order_id
LEFT JOIN (SELECT DISTINCT order_id FROM order_payments) AS p ON o.order_id = p.order_id
LEFT JOIN (SELECT DISTINCT order_id FROM order_reviews)  AS r ON o.order_id = r.order_id
WHERE i.order_id IS NULL
GROUP BY o.order_status
ORDER BY order_cnt DESC
""").df()

,order_status,order_cnt,has_payment,has_review
0,unavailable,603,603,589
1,canceled,164,164,161
2,created,5,5,3
3,invoiced,2,2,2
4,shipped,1,1,1


`delivered` 는 0 건. `unavailable` 603, `canceled` 164, `created` 5, `invoiced` 2,
`shipped` 1 로 767 건(99%)이 상품 미확보로 무산된 주문.

775 건 전부 결제 기록은 있고 756 건에는 리뷰까지 달림.
결제까지 마쳤으나 상품을 받지 못한 고객이 남긴 리뷰.

배송 완료 주문만 쓰는 분석에서는 `delivered` 필터만으로 전부 제외.
`shipped` 인데 품목이 없는 1 건은 앞뒤가 맞지 않는 이상 케이스.

한 주문의 품목이 여러 판매자에 걸치는지 확인.

In [13]:
con.execute("""
SELECT sellers_per_order,
       COUNT(*) AS order_cnt
FROM (
    SELECT order_id,
           COUNT(DISTINCT seller_id) AS sellers_per_order
    FROM order_items
    GROUP BY order_id
)
GROUP BY sellers_per_order
ORDER BY sellers_per_order
""").df()

,sellers_per_order,order_cnt
0,1,97388
1,2,1219
2,3,54
3,4,3
4,5,2


97,388 건은 판매자 1명. 2명 1,219 건, 3명 54 건, 4명 3 건, 5명 2 건으로
여러 판매자에 걸친 주문이 1,278 건(1.3%).

`orders` 의 배송완료일은 주문당 하나뿐이고 품목별 도착 시각은 데이터에 없음.
판매자가 여럿인 주문도 배송 소요 시간은 주문 단위로만 산출 가능.

## 4. order_payments

한 주문에 결제가 여러 건 발생 가능(할부, 바우처 분할 결제). 컬럼 구성 확인.

In [14]:
con.execute("""
SELECT *
FROM order_payments
LIMIT 3
""").df()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


결제 수단(`payment_type`)·할부 개월(`payment_installments`)·금액(`payment_value`)과
주문 내 순번(`payment_sequential`) 보유.

In [15]:
con.execute("""
SELECT COUNT(*)                                          AS row_cnt,
       COUNT(DISTINCT order_id)                          AS order_id_cnt,
       COUNT(DISTINCT (order_id, payment_sequential))    AS composite_key_cnt
FROM order_payments
""").df()

,row_cnt,order_id_cnt,composite_key_cnt
0,103886,99440,103886


복합 키 `(order_id, payment_sequential)` 이 행 수 103,886 과 일치.

결제가 여러 건인 주문의 실제 행 확인.

In [16]:
con.execute("""
SELECT order_id,
       payment_sequential,
       payment_type,
       payment_installments,
       payment_value
FROM order_payments
WHERE order_id IN (
    SELECT order_id
    FROM order_payments
    GROUP BY order_id
    HAVING COUNT(*) > 2
    ORDER BY order_id
    LIMIT 2
)
ORDER BY order_id, payment_sequential
""").df()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,009ac365164f8e06f59d18a08045f6c4,1,credit_card,1,0.88
1,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
2,009ac365164f8e06f59d18a08045f6c4,3,voucher,1,8.25
3,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45
4,009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75
5,009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
6,00bd50cdd31bd22e9081e6e2d5b3577b,1,credit_card,1,4.88
7,00bd50cdd31bd22e9081e6e2d5b3577b,2,voucher,1,40.46
8,00bd50cdd31bd22e9081e6e2d5b3577b,3,voucher,1,40.46


신용카드 1 건에 바우처 여러 건이 붙는 분할 결제.
`payment_sequential` 은 상품이나 수단의 구분이 아니라 주문 안의 결제 순번.

그레인은 한 주문의 결제 수단 한 건. 주문 금액이 필요하면 `order_id` 로
`SUM(payment_value)` 집계.

`order_id` 는 99,440 으로 orders 보다 1 건 부족. 그 1 건을 확인.

In [17]:
con.execute("""
SELECT o.order_id,
       o.order_status,
       o.order_purchase_timestamp,
       o.order_delivered_customer_date
FROM orders AS o
LEFT JOIN (SELECT DISTINCT order_id FROM order_payments) AS p
       ON o.order_id = p.order_id
WHERE p.order_id IS NULL
""").df()

,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,2016-11-09 07:47:38


`delivered` 상태이고 2016-09-15 구매 → 2016-11-09 배송완료.
배송이 정상 완료된 주문인데 결제 기록만 없는 누락.
2016 년 9 월은 이 데이터셋에서 가장 이른 시기로, 초기 적재 결함으로 추정.

배송·평점 분석은 결제 컬럼을 쓰지 않으므로 이 주문을 제외하지 않음.
금액을 다루는 분석으로 확장할 때만 문제.

### 4-1. 운임의 단위

3 에서 유보한 운임을 여기서 판정. `order_items` 의 `freight_value` 가 행마다 붙는 값인지,
주문당 한 번 붙는 값이 행마다 복사된 것인지에 따라 주문 단위 운임의 집계 방법이 갈림.

먼저 한 주문 안에서 운임 값이 변하는지 확인.
주문 단위 값이 복사된 것이라면 한 주문의 모든 행이 같은 값을 가져야 함.

In [18]:
con.execute("""
SELECT COUNT(*)                                     AS multi_item_orders,
       COUNT(*) FILTER (WHERE freight_variants = 1) AS same_freight,
       COUNT(*) FILTER (WHERE freight_variants > 1) AS varying_freight
FROM (
    SELECT order_id,
           COUNT(DISTINCT freight_value) AS freight_variants
    FROM order_items
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
""").df()

,multi_item_orders,same_freight,varying_freight
0,9803,7773,2030


복수 품목 주문 9,803 건 중 운임이 한 값뿐인 주문이 7,773 건, 여러 값인 주문이 2,030 건.
주문 단위 값이 행마다 복사된 것이라면 나올 수 없는 결과이므로 운임은 행에 붙는 값.

다만 값이 변한다는 것만으로 `SUM` 이 주문의 총 운임이라는 판정까지는 되지 않음.
행마다 붙은 값의 합이 실제 총 운임인지 따로 확인이 필요.

`order_payments` 의 결제액과 대조. 같은 주문을 두 방식으로 계산해 어느 쪽이 결제액과
일치하는지 확인.

- `SUM(price) + SUM(freight_value)` 가 일치하면 행마다 붙는 값
- `SUM(price) + MAX(freight_value)` 가 일치하면 주문당 한 번 붙는 값

`order_payments` 는 주문당 여러 행이므로 그대로 조인하면 행이 불어남.
양쪽 모두 `order_id` 로 먼저 집계한 뒤 연결.

In [19]:
con.execute("""
WITH item AS (
    SELECT order_id,
           SUM(price)         AS price_sum,
           SUM(freight_value) AS freight_sum,
           MAX(freight_value) AS freight_max
    FROM order_items
    GROUP BY order_id
),
pay AS (
    SELECT order_id,
           SUM(payment_value) AS paid
    FROM order_payments
    GROUP BY order_id
)
SELECT COUNT(*) AS orders_compared,
       COUNT(*) FILTER (
           WHERE ABS(i.price_sum + i.freight_sum - p.paid) < 0.01
       ) AS sum_matches_payment,
       COUNT(*) FILTER (
           WHERE ABS(i.price_sum + i.freight_max - p.paid) < 0.01
       ) AS max_matches_payment
FROM item AS i
JOIN pay  AS p ON i.order_id = p.order_id
""").df()

,orders_compared,sum_matches_payment,max_matches_payment
0,98665,98278,88666


`SUM` 쪽이 98,278 건, `MAX` 쪽이 88,666 건 일치. 다만 이 비교만으로는 판정이 약함 —
품목이 하나뿐인 주문은 `SUM` 과 `MAX` 가 같은 값이라 양쪽 모두 맞기 때문.

두 방식이 실제로 갈리는 것은 복수 품목 주문뿐이므로 품목 수로 나눠서 다시 확인.

In [20]:
con.execute("""
WITH item AS (
    SELECT order_id,
           COUNT(*)           AS item_cnt,
           SUM(price)         AS price_sum,
           SUM(freight_value) AS freight_sum,
           MAX(freight_value) AS freight_max
    FROM order_items
    GROUP BY order_id
),
pay AS (
    SELECT order_id,
           SUM(payment_value) AS paid
    FROM order_payments
    GROUP BY order_id
)
SELECT CASE WHEN i.item_cnt = 1 THEN '1. 단일 품목'
            ELSE                    '2. 복수 품목' END AS order_type,
       COUNT(*) AS orders,
       COUNT(*) FILTER (
           WHERE ABS(i.price_sum + i.freight_sum - p.paid) < 0.01
       ) AS sum_matches,
       COUNT(*) FILTER (
           WHERE ABS(i.price_sum + i.freight_max - p.paid) < 0.01
       ) AS max_matches
FROM item AS i
JOIN pay  AS p ON i.order_id = p.order_id
GROUP BY order_type
ORDER BY order_type
""").df()

,order_type,orders,sum_matches,max_matches
0,1. 단일 품목,88863,88627,88627
1,2. 복수 품목,9802,9651,39


단일 품목 주문은 두 방식이 같은 값이라 88,627 로 동일. 판정에 쓸 수 없음.
방식이 갈리는 복수 품목 9,802 건에서 `SUM` 은 9,651 건(98.5 %), `MAX` 는 39 건(0.4 %).

운임은 행마다 붙는 값이며 주문 단위 운임은 `SUM(freight_value)`.
3 에서 유보한 규칙을 여기서 확정.

앞 셀의 복수 품목 9,803 건이 여기서 9,802 건인 것은 결제 기록이 없는 주문 1 건이
조인에서 빠졌기 때문. 그 주문이 복수 품목 주문이었다.

상품 금액과 운임의 합이 결제액과 일치하지 않는 주문이 387 건 잔존(단일 236, 복수 151).
두 방식 모두에서 불일치하므로 운임의 단위 판정에는 영향이 없고, 원인은 결제 쪽에 존재.
금액을 다루는 분석으로 확장할 때 별도 확인 대상.

## 5. order_reviews

배송 시간과 평점을 연결하려면 `orders` 와 조인 필요.
한 주문에 리뷰가 몇 건 달리는지 확인. 먼저 컬럼 구성.

In [21]:
con.execute("""
SELECT *
FROM order_reviews
LIMIT 3
""").df()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24


평점(`review_score`)과 리뷰 제목·본문, 작성일(`review_creation_date`)과
응답 시각(`review_answer_timestamp`) 보유.

In [22]:
con.execute("""
SELECT COUNT(*)                                 AS row_cnt,
       COUNT(DISTINCT review_id)                AS review_id_cnt,
       COUNT(DISTINCT order_id)                 AS order_id_cnt,
       COUNT(DISTINCT (review_id, order_id))    AS composite_key_cnt
FROM order_reviews
""").df()

,row_cnt,review_id_cnt,order_id_cnt,composite_key_cnt
0,99224,98410,98673,99224


`review_id` 98,410, `order_id` 98,673 으로 어느 쪽도 단독 키가 아님.
두 열의 조합만 행 수 99,224 와 일치.

중복이 양방향이므로 각각의 분포 확인.

In [23]:
con.execute("""
SELECT reviews_per_order,
       COUNT(*) AS order_cnt
FROM (
    SELECT order_id,
           COUNT(*) AS reviews_per_order
    FROM order_reviews
    GROUP BY order_id
)
GROUP BY reviews_per_order
ORDER BY reviews_per_order
""").df()

,reviews_per_order,order_cnt
0,1,98126
1,2,543
2,3,4


In [24]:
con.execute("""
SELECT orders_per_review,
       COUNT(*) AS review_cnt
FROM (
    SELECT review_id,
           COUNT(*) AS orders_per_review
    FROM order_reviews
    GROUP BY review_id
)
GROUP BY orders_per_review
ORDER BY orders_per_review
""").df()

,orders_per_review,review_cnt
0,1,97621
1,2,764
2,3,25


리뷰가 2 건 달린 주문 543 건, 3 건 달린 주문 4 건.
서로 다른 2 개 주문에 걸친 리뷰 764 건, 3 개 주문에 걸친 리뷰 25 건.

리뷰가 여러 건인 주문의 실제 모습 확인.

In [25]:
con.execute("""
SELECT r.order_id,
       r.review_id,
       r.review_score,
       r.review_creation_date
FROM order_reviews AS r
JOIN (
    SELECT order_id
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY order_id
    LIMIT 3
) AS d
  ON r.order_id = d.order_id
ORDER BY r.order_id, r.review_creation_date
""").df()

,order_id,review_id,review_score,review_creation_date
0,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25
1,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29
2,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22
3,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04
4,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30
5,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30


같은 주문에 `review_id` 와 작성일이 서로 다른 리뷰가 달려 있음.

이 리뷰들의 평점이 서로 일치하는지 전체 규모로 확인.

In [26]:
con.execute("""
SELECT score_gap,
       COUNT(*) AS order_cnt
FROM (
    SELECT order_id,
           MAX(review_score) - MIN(review_score) AS score_gap
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
GROUP BY score_gap
ORDER BY score_gap
""").df()

,score_gap,order_cnt
0,0,345
1,1,90
2,2,47
3,3,32
4,4,33


리뷰가 여러 건인 주문 547 건 중 345 건은 평점이 동일. 나머지 202 건은 평점이 갈리며,
그중 33 건은 차이가 4 로 1 점과 5 점이 한 주문에 함께 존재.

평균으로 합치면 정반대 평가가 중간값으로 뭉개짐.

중복이 재입력인지 재작성인지는 두 리뷰의 시간 간격으로 갈림.
`review_creation_date` 는 시각이 없는 날짜라 자정을 넘기면 몇 분 차이도 다른 날이 되므로,
시각이 있는 `review_answer_timestamp` 를 사용. 두 기준의 차이를 먼저 확인.

In [27]:
con.execute("""
SELECT COUNT(*) AS same_score_orders,
       COUNT(*) FILTER (WHERE date_gap = 0)                  AS date_says_same_day,
       COUNT(*) FILTER (WHERE date_gap > 0 AND gap_min < 60) AS date_says_diff_but_under_1h
FROM (
    SELECT order_id,
           MAX(review_score) - MIN(review_score) AS score_gap,
           date_diff('day', MIN(review_creation_date),
                            MAX(review_creation_date)) AS date_gap,
           date_diff('minute', MIN(review_answer_timestamp),
                               MAX(review_answer_timestamp)) AS gap_min
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
WHERE score_gap = 0
""").df()

,same_score_orders,date_says_same_day,date_says_diff_but_under_1h
0,345,125,37


날짜 기준으로는 125 건이 "같은 날"이지만, "다른 날"로 잡힌 것 중 37 건은
응답 시각 차이가 1 시간 미만. 자정을 넘겼을 뿐 동시 입력에 가까움.
날짜 컬럼만으로 분류하면 이 37 건을 놓침.

중복 입력이라면 평점 외에 제목과 본문까지 같을 수 있음.
다만 응답 시각을 비교에 넣으면 시각이 조금만 달라도 다른 행이 되어 중복을 찾을 수 없음.
응답 시각이 실제로 서로 다른지부터 확인.

In [28]:
con.execute("""
SELECT COUNT(*)                               AS same_score_orders,
       COUNT(*) FILTER (WHERE ts_variants = 1) AS same_answer_ts
FROM (
    SELECT order_id,
           MAX(review_score) - MIN(review_score)   AS score_gap,
           COUNT(DISTINCT review_answer_timestamp) AS ts_variants
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
WHERE score_gap = 0
""").df()

,same_score_orders,same_answer_ts
0,345,0


응답 시각이 같은 주문은 0 건 — 345 건 전부 응답 시각이 다름.
응답 시각을 비교에 넣으면 어떤 두 행도 같아지지 않으므로, 시각을 뺀 나머지가 같은지를 봐야 함.

내용이 같은지와 시간 간격을 교차해서 확인.

In [29]:
con.execute("""
SELECT content_same,
       COUNT(*)                                                 AS order_cnt,
       COUNT(*) FILTER (WHERE gap_min < 60)                     AS under_1h,
       COUNT(*) FILTER (WHERE gap_min >= 60 AND gap_min < 1440) AS h1_to_1d,
       COUNT(*) FILTER (WHERE gap_min >= 1440)                  AS over_1d
FROM (
    SELECT order_id,
           MAX(review_score) - MIN(review_score) AS score_gap,
           COUNT(DISTINCT COALESCE(review_comment_title, '')) = 1
             AND COUNT(DISTINCT COALESCE(review_comment_message, '')) = 1 AS content_same,
           date_diff('minute', MIN(review_answer_timestamp),
                               MAX(review_answer_timestamp)) AS gap_min
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
)
WHERE score_gap = 0
GROUP BY content_same
ORDER BY content_same DESC
""").df()

,content_same,order_cnt,under_1h,h1_to_1d,over_1d
0,True,225,93,25,107
1,False,120,29,9,82


내용까지 같은 주문 225 건, 내용이 다른 주문 120 건.

내용이 같으면서 응답 시각 차이가 1 시간 미만인 93 건은 같은 입력이 두 번 기록된 쪽,
1 일 이상 벌어진 107 건은 앞서 남긴 것을 잊고 다시 입력한 쪽에 가까움.
어느 쪽이든 평점이 같으므로 평점 분석에서는 구분이 불필요.

반대 방향, 즉 한 `review_id` 가 여러 주문에 걸친 경우의 실제 모습 확인.

In [30]:
con.execute("""
SELECT r.review_id,
       r.order_id,
       r.review_score,
       r.review_creation_date
FROM order_reviews AS r
JOIN (
    SELECT review_id
    FROM order_reviews
    GROUP BY review_id
    HAVING COUNT(*) > 1
    ORDER BY review_id
    LIMIT 3
) AS d
  ON r.review_id = d.review_id
ORDER BY r.review_id, r.order_id
""").df()

,review_id,order_id,review_score,review_creation_date
0,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07
1,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07
2,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21
3,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21
4,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07
5,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07


같은 `review_id` 가 서로 다른 주문에 붙어 있고 점수와 작성일은 동일.
한 번의 응답이 여러 주문을 함께 평가한 형태.

리뷰가 주문이 아니라 상품 단위로 달릴 가능성도 확인.
상품 단위라면 품목이 많은 주문일수록 리뷰 건수가 비례해 늘어야 함.

In [31]:
con.execute("""
SELECT r.reviews_per_order,
       COUNT(*) AS order_cnt,
       ROUND(AVG(COALESCE(i.item_cnt, 0)), 2) AS avg_item_cnt
FROM (
    SELECT order_id,
           COUNT(*) AS reviews_per_order
    FROM order_reviews
    GROUP BY order_id
) AS r
LEFT JOIN (
    SELECT order_id,
           COUNT(*) AS item_cnt
    FROM order_items
    GROUP BY order_id
) AS i
  ON r.order_id = i.order_id
GROUP BY r.reviews_per_order
ORDER BY r.reviews_per_order
""").df()

,reviews_per_order,order_cnt,avg_item_cnt
0,1,98126,1.13
1,2,543,1.20
2,3,4,1.50


리뷰 1 건인 주문의 평균 품목은 1.13 개, 2 건인 주문은 1.20 개, 3 건인 주문은 1.50 개.
품목 수가 리뷰 건수를 설명하지 못함. `order_reviews` 에 `product_id` 컬럼도 없음.
리뷰는 상품이 아니라 주문에 달림.

그레인은 리뷰와 주문의 연결 한 건, 키는 `(review_id, order_id)`.

`orders` 에 리뷰를 그대로 조인하면 리뷰가 달린 주문 98,673 건이 99,224 행으로 증가.
551 행이 늘고, 리뷰가 여러 개인 주문은 평균 평점에서 가중치를 더 가짐.
배송 시간과 평점을 볼 때는 조인 전에 주문 단위로 평점을 정리.
처리 기준은 분석 표본을 정할 때 결정.

## 6. products

다른 테이블이 참조하는 마스터. 컬럼 구성 확인.

In [32]:
con.execute("""
SELECT *
FROM products
LIMIT 3
""").df()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15


카테고리명과 이름·설명 길이, 사진 수, 무게·치수 보유.

`product_id` 가 키인지 확인.

In [33]:
con.execute("""
SELECT COUNT(*) AS row_cnt,
       COUNT(DISTINCT product_id) AS product_id_cnt
FROM products
""").df()

,row_cnt,product_id_cnt
0,32951,32951


32,951 로 동일. 그레인은 상품 1개, 키는 `product_id`.

## 7. sellers

컬럼 구성 확인.

In [34]:
con.execute("""
SELECT *
FROM sellers
LIMIT 3
""").df()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


판매자의 주·도시·우편번호 앞자리 보유. `customers` 와 같은 구성.

`seller_id` 가 키인지 확인.

In [35]:
con.execute("""
SELECT COUNT(*) AS row_cnt,
       COUNT(DISTINCT seller_id) AS seller_id_cnt
FROM sellers
""").df()

,row_cnt,seller_id_cnt
0,3095,3095


3,095 로 동일. 그레인은 판매자 1명, 키는 `seller_id`.

## 8. product_category_name_translation

카테고리명을 영문으로 옮기는 대응표. 컬럼 구성 확인.

In [36]:
con.execute("""
SELECT *
FROM product_category_name_translation
LIMIT 3
""").df()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


포르투갈어 카테고리명과 영문 대응 두 열.

`product_category_name` 이 키인지 확인.

In [37]:
con.execute("""
SELECT COUNT(*) AS row_cnt,
       COUNT(DISTINCT product_category_name) AS category_name_cnt
FROM product_category_name_translation
""").df()

,row_cnt,category_name_cnt
0,71,71


71 로 동일. 그레인은 카테고리명 1개.

행이 늘지는 않지만 짝이 없을 수는 있으므로,
`products` 의 카테고리가 이 71 개에 모두 있는지 확인.

In [38]:
con.execute("""
SELECT p.product_category_name,
       COUNT(*) AS product_cnt
FROM products AS p
LEFT JOIN product_category_name_translation AS t
       ON p.product_category_name = t.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND t.product_category_name IS NULL
GROUP BY p.product_category_name
ORDER BY product_cnt DESC
""").df()

,product_category_name,product_cnt
0,portateis_cozinha_e_preparadores_de_alimentos,10
1,pc_gamer,3


`portateis_cozinha_e_preparadores_de_alimentos` 와 `pc_gamer` 두 카테고리가 번역표에 없음.
해당 상품 13 개.

카테고리명을 영문으로 바꿔 집계하면 이 13 개가 NULL 로 빠지므로,
`LEFT JOIN` 후 번역이 없으면 원문 카테고리명을 그대로 사용.

## 9. geolocation

컬럼 구성 확인.

In [39]:
con.execute("""
SELECT *
FROM geolocation
LIMIT 3
""").df()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.545621,-46.639292,sao paulo,SP
1,01046,-23.546081,-46.644820,sao paulo,SP
2,01046,-23.546129,-46.642951,sao paulo,SP


우편번호 앞자리와 위경도, 도시·주로 구성. 행을 식별할 ID 컬럼 없음.

우편번호 앞자리와 전체 컬럼 조합, 두 가지로 유일성 확인.

In [40]:
con.execute("""
SELECT COUNT(*) AS row_cnt,
       COUNT(DISTINCT geolocation_zip_code_prefix) AS zip_prefix_cnt,
       COUNT(DISTINCT (geolocation_zip_code_prefix,
                       geolocation_lat,
                       geolocation_lng,
                       geolocation_city,
                       geolocation_state)) AS all_column_cnt
FROM geolocation
""").df()

,row_cnt,zip_prefix_cnt,all_column_cnt
0,1000163,19015,738332


우편번호 앞자리는 19,015 개. 전체 컬럼을 조합해도 738,332 로 행 수 1,000,163 에 미달.
완전 중복 행이 261,831 건이며 자연 키가 부재.

이 프로젝트는 좌표나 거리를 다루지 않고, 지역 정보는 `customers` 와 `sellers` 가
주·도시를 자체 보유. `geolocation` 은 사용하지 않음.

## 10. 참조 무결성

앞의 절들은 `orders` 를 기준으로 두고 주문에 품목·결제·리뷰가 있는지를 봄.
반대 방향, 즉 자식 테이블의 행이 가리키는 부모가 실제로 있는지는 아직 보지 않음.

CSV 를 그대로 적재해 외래 키 제약이 없으므로 짝이 없는 값이 들어 있어도 DB 가 알려주지 않음.
짝이 없는 행은 조인할 때 오류 없이 결과에서 빠지므로, 집계 결과가 틀리는 대신
표본이 말없이 줄어듦.

깨져 있다고 반드시 결함인 것은 아님. 단종된 상품이 마스터에서 지워졌을 수도 있음.
확인하려는 것은 오류가 아니라 조인할 때 빠지는 규모.

다섯 관계를 안티조인으로 한 번에 셈.
`customers`↔`orders` 는 2 에서 양방향으로 확인했고,
`products`↔번역표는 8 에서 확인했으므로 제외.
`geolocation` 은 사용하지 않으므로 우편번호 참조도 대상 아님.

In [41]:
con.execute("""
SELECT '1. order_items → orders' AS reference,
       COUNT(*)                  AS orphan_rows
FROM order_items AS i
LEFT JOIN orders AS o ON i.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL
SELECT '2. order_payments → orders', COUNT(*)
FROM order_payments AS p
LEFT JOIN orders AS o ON p.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL
SELECT '3. order_reviews → orders', COUNT(*)
FROM order_reviews AS r
LEFT JOIN orders AS o ON r.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL
SELECT '4. order_items → products', COUNT(*)
FROM order_items AS i
LEFT JOIN products AS p ON i.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL
SELECT '5. order_items → sellers', COUNT(*)
FROM order_items AS i
LEFT JOIN sellers AS s ON i.seller_id = s.seller_id
WHERE s.seller_id IS NULL

ORDER BY reference
""").df()

,reference,orphan_rows
0,1. order_items → orders,0
1,2. order_payments → orders,0
2,3. order_reviews → orders,0
3,4. order_items → products,0
4,5. order_items → sellers,0


다섯 관계 모두 0 건. 자식 테이블의 행이 가리키는 부모는 예외 없이 존재하며,
`order_items` 를 `products` 나 `sellers` 에 붙여도 112,650 행이 그대로 유지됨.

짝이 없던 경우는 전부 반대 방향이었음 — 품목 없는 주문 775 건, 결제 없는 주문 1 건,
리뷰 없는 주문. 주문이 취소되면 품목이 없을 수 있으나 품목이 있는데 주문이 없을 수는 없으므로
이 비대칭은 자연스러움.

다만 자식에서 부모를 찾는 방향이 항상 온전한 것은 아님.
8 에서 확인한 `products` 에서 번역표를 찾는 방향은 13 개가 깨져 있었다.
번역표는 거래 과정에서 생긴 것이 아니라 따로 만든 대응표라 상품 카테고리를 다 담지 못함.

## 11. 정리

| 테이블 | 한 행의 의미 | 키 | 행 수 |
|---|---|---|---|
| orders | 주문 1건 | `order_id` | 99,441 |
| customers | 주문 시점의 고객 레코드 | `customer_id` | 99,441 |
| order_items | 주문에서 구매한 상품 1개 | `(order_id, order_item_id)` | 112,650 |
| order_payments | 한 주문의 결제 수단 한 건 | `(order_id, payment_sequential)` | 103,886 |
| order_reviews | 리뷰와 주문의 연결 한 건 | `(review_id, order_id)` | 99,224 |
| products | 상품 1개 | `product_id` | 32,951 |
| sellers | 판매자 1명 | `seller_id` | 3,095 |
| product_category_name_translation | 카테고리명 1개 | `product_category_name` | 71 |
| geolocation | 우편번호 구간의 좌표 관측치 1건 | 없음 (완전 중복 존재) | 1,000,163 |

**조인 규칙**

- `orders` 를 기준으로 두고 `order_items`·`order_payments`·`order_reviews` 는
  `order_id` 로 집계 후 연결
- `orders`↔`customers` 는 `customer_id` 로 1:1, 집계 없이 바로 연결
- 도시 단위 집계는 `(customer_city, customer_state)` 조합으로
- `products`↔번역표는 `LEFT JOIN`, 번역이 없으면 원문 카테고리명 사용
- `order_items`↔`products`·`sellers` 는 짝이 없는 행이 없어 조인해도 행이 빠지지 않음
- `geolocation` 은 사용하지 않음
- 사람 단위 지표는 `customer_unique_id` 로 집계

**다음 노트북으로 넘길 것**

- 리뷰가 여러 건인 주문 547 건 — 점수가 같은 345 건과 갈리는 202 건의 처리 기준
- 리뷰가 없는 주문 — 배송·평점 분석의 표본 범위
- 리뷰 텍스트 결측 규모
- 상품을 받지 못한 주문 756 건에 달린 리뷰 — 낮은 평점의 원인을 볼 때 별도 관찰 대상
- 지역 단위를 주로 둘지 도시로 내릴지

## 12. 연결 종료

DuckDB 파일은 한 프로세스만 쓰기 모드로 열 수 있으므로 다음 노트북을 위해 여기서 종료.

In [42]:
con.close()